# SIC 2026 · Inteligencia Artificial
## Clase 15: Keras aplicado, con casos reales (Capítulo 8, Unidad 3)

**Samsung Innovation Campus Chile 2026 · ONG Innovacien**

En la Clase 14 llegamos hasta la lámina 152. Hoy vemos **Keras a fondo**, pero aprendiendo con **casos reales** que ya conocen, para comparar las redes neuronales con lo que hicieron en scikit-learn:

| Hora | Parte | Caso | Tipo de problema |
|---|---|---|---|
| 18:00 a 18:25 | 1. Keras paso a paso | | Los 5 pasos y cómo contar parámetros |
| 18:25 a 19:25 | 2. Fuga de clientes | Telco Customer Churn (Clase 6) | Clasificación binaria desbalanceada |
| 19:25 a 19:35 | Pausa | | |
| 19:35 a 20:15 | 3. Precio de diamantes | Diamonds (Clase 4) | Regresión |
| 20:15 a 20:40 | 4. Gama de precio | Diamonds (Clase 4) | Clasificación multiclase |
| 20:40 a 21:20 | 5. Prendas de ropa | Fashion-MNIST | Imágenes con CNN |
| 21:20 a 21:55 | 6. Pasajeros de avión | Flights | Series de tiempo con RNN, LSTM y GRU |
| Para la casa | 7. Desafío | Telco Customer Churn | Superar al mejor modelo |

**Hay muchos ejercicios (16 más el desafío).** La idea es que ustedes escriban el código. Cada ejercicio tiene su verificación y las soluciones están al final.

**Antes de empezar:** *Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU (T4)*.

In [ ]:
!pip install -q kagglehub

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.metrics import (roc_auc_score, recall_score, precision_score, f1_score,
                             accuracy_score, mean_absolute_error, classification_report, confusion_matrix)

def revisar(condicion, mensaje_ok="Correcto.", mensaje_error="Todavía no. Revisa tu respuesta."):
    try:
        print(mensaje_ok if condicion() else mensaje_error)
    except Exception:
        print(mensaje_error)

def graficar(historia, metricas=("loss",)):
    fig, ejes = plt.subplots(1, len(metricas), figsize=(5.5 * len(metricas), 3.5))
    ejes = np.atleast_1d(ejes)
    for ax, m in zip(ejes, metricas):
        ax.plot(historia.history[m], label="entrenamiento")
        if "val_" + m in historia.history:
            ax.plot(historia.history["val_" + m], label="validación")
        ax.set_title(m); ax.set_xlabel("Época"); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "no")

---
# Parte 1. Keras paso a paso (láminas 154 a 163)

En la Clase 14 escribimos a mano el ciclo de entrenamiento: predecir, calcular el error, calcular gradientes con `GradientTape` y actualizar. **Keras hace todo ese ciclo con `fit`**. El trabajo se ordena en 5 pasos:

| Paso | Código | Qué decide |
|---|---|---|
| 1. Definir | `modelo = keras.Sequential([...])` | Las capas: cuántas, de qué tipo, cuántas neuronas, qué activación |
| 2. Compilar | `modelo.compile(optimizer=..., loss=..., metrics=[...])` | Cómo aprende (optimizador), qué minimiza (pérdida) y qué reporta (métricas) |
| 3. Entrenar | `modelo.fit(X, y, epochs=..., batch_size=..., validation_data=...)` | Cuántas pasadas por los datos y de a cuántos ejemplos |
| 4. Evaluar | `modelo.evaluate(X_prueba, y_prueba)` | La pérdida y las métricas en datos nuevos |
| 5. Predecir | `modelo.predict(X_nuevo)` | Las salidas de la red para casos nuevos |

## Paso 1.1. La regla de oro: la salida y la pérdida dependen del problema

| Problema | Última capa | Pérdida | Lo usaremos en |
|---|---|---|---|
| Regresión (un número) | `Dense(1)`, sin activación | `"mse"` o `"mae"` | Parte 3 |
| Binaria (sí o no) | `Dense(1, activation="sigmoid")` | `"binary_crossentropy"` | Parte 2 |
| Multiclase, etiquetas enteras | `Dense(k, activation="softmax")` | `"sparse_categorical_crossentropy"` | Partes 4 y 5 |
| Multiclase, etiquetas one-hot | `Dense(k, activation="softmax")` | `"categorical_crossentropy"` | |

En las capas ocultas: `relu`. Optimizador: `"adam"` como punto de partida.

## Paso 1.2. Los argumentos de `fit`

| Argumento | Qué es | Valor típico |
|---|---|---|
| `epochs` | Cuántas veces se recorren todos los datos de entrenamiento | 20 a 200 (con parada temprana) |
| `batch_size` | De a cuántos ejemplos se calcula cada actualización | 32, 64 o 128 |
| `validation_data=(X_val, y_val)` | Datos para vigilar el sobreajuste mientras entrena | Un 15% a 20% aparte |
| `class_weight={0: p0, 1: p1}` | Peso de los errores de cada clase, para datos desbalanceados (Clase 6) | |
| `callbacks=[...]` | Acciones automáticas durante el entrenamiento | `EarlyStopping` |
| `verbose` | 0 no muestra nada, 1 muestra una barra por época | |

> **Parada temprana (`EarlyStopping`):** vigila una métrica de validación y **detiene el entrenamiento** cuando deja de mejorar durante `patience` épocas. Con `restore_best_weights=True`, se queda con los pesos de la mejor época. Así no hay que adivinar cuántas épocas usar, y se evita el sobreajuste.

## Paso 1.3. Contar parámetros

Una capa `Dense` con $e$ entradas y $n$ neuronas tiene **$e \\times n + n$** parámetros: un peso por cada entrada en cada neurona, más un sesgo por neurona. `Dropout` no tiene parámetros: solo apaga neuronas.

In [ ]:
ejemplo = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(4, activation="relu"),       # 10 x 4 + 4 = 44
    layers.Dense(1, activation="sigmoid"),    # 4 x 1 + 1 = 5
])
ejemplo.summary()

**Qué deberías ver:** 44 y 5 parámetros, 49 en total.

### Ejercicio 1
En la Parte 2, la red recibirá **46** columnas (después del one-hot). ¿Cuántos parámetros tiene una primera capa `Dense(32)`? Guárdalo en `params_capa1`.

In [ ]:
params_capa1 = None  # TU CÓDIGO AQUÍ
print(params_capa1)

In [ ]:
revisar(lambda: params_capa1 == 46 * 32 + 32)

### Ejercicio 2
Calcula el **total** de parámetros de esta red: 46 entradas, `Dense(32)`, `Dropout(0.3)`, `Dense(16)`, `Dense(1)`. Guárdalo en `params_total` y compruébalo construyendo el modelo y usando `modelo.count_params()`.

In [ ]:
params_total = None  # TU CÓDIGO AQUÍ
print(params_total)

In [ ]:
revisar(lambda: params_total == (46 * 32 + 32) + (32 * 16 + 16) + (16 * 1 + 1))

---
# Parte 2. Caso: fuga de clientes con Keras (Telco Customer Churn)

Es el caso de la Clase 6: una compañía de telecomunicaciones quiere identificar a los clientes con riesgo de irse. En esa clase, la mejor opción fue una **regresión logística balanceada** (ROC AUC cercano a 0,84). Hoy la pregunta es: **¿una red neuronal lo hace mejor?**

## Paso 2.1. Cargar y preparar

Igual que en la Clase 6: `TotalCharges` a número, `customerID` fuera y `Churn` a 0 y 1. Esta vez usamos **todas** las columnas.

In [ ]:
def cargar_telco():
    try:
        import kagglehub
        carpeta = kagglehub.dataset_download("blastchar/telco-customer-churn")
        return pd.read_csv(os.path.join(carpeta, "WA_Fn-UseC_-Telco-Customer-Churn.csv"))
    except Exception:
        return pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

telco = cargar_telco()
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce").fillna(0)
telco["SeniorCitizen"] = telco["SeniorCitizen"].map({0: "No", 1: "Yes"})

y_t = (telco["Churn"] == "Yes").astype(int).values
X_t = telco.drop(columns=["customerID", "Churn"])
NUM_T = ["tenure", "MonthlyCharges", "TotalCharges"]
CAT_T = [c for c in X_t.columns if c not in NUM_T]
print("Clientes:", len(X_t), "| Tasa de fuga:", round(y_t.mean(), 3), "| Categóricas:", len(CAT_T))

## Paso 2.2. Tres conjuntos: entrenamiento, validación y prueba

En Keras conviene separar **tres** conjuntos:

- **Entrenamiento:** para ajustar los pesos.
- **Validación:** para vigilar el sobreajuste y decidir cuándo parar (`EarlyStopping`).
- **Prueba:** solo al final, una vez.

El preprocesamiento se ajusta **solo con el entrenamiento** y se aplica a los otros dos (Clase 4).

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42, stratify=y_tr)

prep_t = ColumnTransformer([
    ("num", StandardScaler(), NUM_T),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_T),   # Keras necesita una matriz densa
]).fit(X_tr)

A_tr, A_va, A_te = prep_t.transform(X_tr), prep_t.transform(X_va), prep_t.transform(X_te)
print("Entrenamiento:", A_tr.shape, "| Validación:", A_va.shape, "| Prueba:", A_te.shape)

**Qué deberías ver:** 46 columnas después del one-hot, y tres conjuntos de unos 4.500, 1.100 y 1.400 clientes.

## Paso 2.3. Pesos por clase

Solo un 27% de los clientes se va. Igual que `class_weight="balanced"` en scikit-learn, le damos más peso a los errores en la clase minoritaria. La fórmula de "balanceado" es:

$$\\text{peso de la clase } c = \\frac{\\text{total de ejemplos}}{2 \\times \\text{ejemplos de la clase } c}$$

### Ejercicio 3
Calcula `pesos_clase`, un diccionario `{0: ..., 1: ...}` con la fórmula, usando `y_tr`. Convierte los valores a `float`.

In [ ]:
pesos_clase = None  # TU CÓDIGO AQUÍ
print(pesos_clase)

In [ ]:
revisar(lambda: np.isclose(pesos_clase[0], len(y_tr) / (2 * (y_tr == 0).sum())) and np.isclose(pesos_clase[1], len(y_tr) / (2 * (y_tr == 1).sum())))

**Qué deberías ver:** cerca de 0,68 para la clase 0 y 1,88 para la clase 1. Un error con un cliente que se va pesa casi el triple.

## Paso 2.4. Construir, compilar y entrenar

### Ejercicio 4
Completa `construir_red_churn()` para que devuelva esta red **ya compilada**:

- Entrada con la forma de `A_tr` (46 columnas).
- `Dense(32)` con `relu`, luego `Dropout(0.3)`, luego `Dense(16)` con `relu`.
- Salida para clasificación binaria.
- Compilar con `adam`, la pérdida de clasificación binaria y estas métricas: `keras.metrics.AUC(name="auc")` y `keras.metrics.Recall(name="recall")`.

In [ ]:
def construir_red_churn():
    keras.utils.set_random_seed(42)
    modelo = None   # TU CÓDIGO AQUÍ: keras.Sequential([...])
    # TU CÓDIGO AQUÍ: modelo.compile(...)
    return modelo

red_churn = construir_red_churn()
red_churn.summary() if red_churn is not None else None

In [ ]:
revisar(lambda: red_churn.count_params() == 2049 and red_churn.loss == "binary_crossentropy" and red_churn.layers[-1].units == 1)

Ahora entrenamos con parada temprana: vigilamos el AUC de validación (`val_auc`), que queremos **maximizar**, con paciencia de 10 épocas.

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=10, restore_best_weights=True)
historia_churn = red_churn.fit(A_tr, y_tr, validation_data=(A_va, y_va), epochs=100, batch_size=64,
                               class_weight=pesos_clase, callbacks=[parada], verbose=0)
print("Épocas entrenadas:", len(historia_churn.history["loss"]), "(se detuvo sola)")
graficar(historia_churn, ("loss", "auc"))

**Qué deberías ver:** se detiene sola cerca de la época 25. La pérdida de entrenamiento sigue bajando, pero la de validación se estanca: desde ahí, la red empezaría a sobreajustar.

## Paso 2.5. Evaluar con prueba

In [ ]:
prob_churn = red_churn.predict(A_te, verbose=0).ravel()
pred_churn = (prob_churn >= 0.5).astype(int)
print(f"ROC AUC en prueba: {roc_auc_score(y_te, prob_churn):.3f}")
print(classification_report(y_te, pred_churn, target_names=["Se queda", "Se va"], digits=3))

**Qué deberías ver:** un ROC AUC cercano a 0,84 y un recall de la clase "Se va" cercano a 0,78.

### Ejercicio 5
Guarda en `recall_red` el recall de la clase "Se va" con umbral 0,5, y en `n_contactar` cuántos clientes de prueba habría que contactar (los marcados como "Se va").

In [ ]:
recall_red = None    # TU CÓDIGO AQUÍ
n_contactar = None   # TU CÓDIGO AQUÍ
print(recall_red, n_contactar)

In [ ]:
revisar(lambda: np.isclose(recall_red, recall_score(y_te, prob_churn >= 0.5)) and n_contactar == int((prob_churn >= 0.5).sum()))

### Ejercicio 6
Con la Clase 7 en mente: busca el umbral, entre `0.30` y `0.80` en pasos de `0.05`, que **maximiza el F1** de la clase "Se va". Guárdalo en `umbral_f1` (redondeado a 2 decimales).

In [ ]:
umbrales = np.round(np.arange(0.30, 0.81, 0.05), 2)
umbral_f1 = None  # TU CÓDIGO AQUÍ
print(umbral_f1)

In [ ]:
_f = [f1_score(y_te, prob_churn >= u) for u in umbrales]
revisar(lambda: np.isclose(umbral_f1, umbrales[int(np.argmax(_f))]))

## Paso 2.6. ¿Le ganó la red a la regresión logística?

### Ejercicio 7
Entrena una `LogisticRegression(max_iter=3000, class_weight="balanced")` con **entrenamiento más validación** (`np.vstack` y `np.concatenate`), calcula su ROC AUC en prueba y guárdalo en `auc_logistica`. Luego guarda en `gana` el texto `"red"` o `"logistica"`, según cuál tenga mayor AUC.

In [ ]:
from sklearn.linear_model import LogisticRegression

auc_logistica = None  # TU CÓDIGO AQUÍ
gana = None           # TU CÓDIGO AQUÍ
print(auc_logistica, gana)

In [ ]:
_lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(np.vstack([A_tr, A_va]), np.concatenate([y_tr, y_va]))
_auc = roc_auc_score(y_te, _lr.predict_proba(A_te)[:, 1])
revisar(lambda: np.isclose(auc_logistica, _auc, atol=1e-3) and gana == ("red" if roc_auc_score(y_te, prob_churn) > _auc else "logistica"))

**Qué deberías ver:** resultados casi idénticos (cerca de 0,84 los dos), y probablemente la logística un poco arriba.

**Esta es la lección central de la parte.** En **datos en tabla** con pocas columnas, una red neuronal rara vez supera a un buen modelo clásico, y cuesta más entrenarla y explicarla. Las redes brillan con **imágenes, texto y sonido**, donde los modelos clásicos no tienen cómo ver la estructura (Parte 5). Para el capstone: si sus datos son una tabla, prueben primero scikit-learn.

---
# Parte 3. Caso: precio de diamantes (regresión)

En la Clase 4 clasificamos diamantes en gamas. Ahora predecimos el **precio exacto**. Usamos la misma muestra de 10.000 diamantes.

## Paso 3.1. Un truco para precios: el logaritmo

Los precios van de 300 a casi 19.000 dólares y están muy cargados hacia los valores bajos. Si la red predice el precio directo, los diamantes caros dominan el error. Predecir el **logaritmo del precio** pone todo en una escala parecida, y después se vuelve a dólares con `np.exp`.

In [ ]:
dm = sns.load_dataset("diamonds").sample(10_000, random_state=42).reset_index(drop=True)
for c in ["cut", "color", "clarity"]:
    dm[c] = dm[c].astype(str)

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.5))
ejes[0].hist(dm["price"], bins=50, color="#1428A0"); ejes[0].set_title("Precio (USD)")
ejes[1].hist(np.log(dm["price"]), bins=50, color="#FF6B35"); ejes[1].set_title("Logaritmo del precio")
plt.show()

F_D = ["carat", "cut", "color", "clarity", "depth", "table"]
ORDENES = [["Fair", "Good", "Very Good", "Premium", "Ideal"], ["J", "I", "H", "G", "F", "E", "D"],
           ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]]
prep_d = ColumnTransformer([("num", "passthrough", ["carat", "depth", "table"]),
                            ("cat", OrdinalEncoder(categories=ORDENES), ["cut", "color", "clarity"])])

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(dm[F_D], np.log(dm["price"].values), test_size=0.2, random_state=42)
esc_d = StandardScaler().fit(prep_d.fit_transform(Xd_tr))
D_tr, D_te = esc_d.transform(prep_d.transform(Xd_tr)), esc_d.transform(prep_d.transform(Xd_te))
precio_real = np.exp(yd_te)
print("Entrenamiento:", D_tr.shape, "| Prueba:", D_te.shape)

**Qué deberías ver:** el histograma del precio muy cargado a la izquierda, y el del logaritmo mucho más parejo.

### Ejercicio 8
Construye, compila y entrena `red_precio`:

- Entrada de 6 columnas, `Dense(64)` y `Dense(32)` con `relu`, y la salida de **regresión**.
- Compilar con `adam`, pérdida `"mse"` y métrica `"mae"`.
- Entrenar con `validation_split=0.2`, hasta 200 épocas, `batch_size=64` y `EarlyStopping(patience=10, restore_best_weights=True)`.

Después, calcula `mae_red` en **dólares**: predice sobre `D_te`, vuelve a dólares con `np.exp` y compara con `precio_real`.

In [ ]:
keras.utils.set_random_seed(42)
red_precio = None  # TU CÓDIGO AQUÍ (definir, compilar y entrenar)

mae_red = None     # TU CÓDIGO AQUÍ
print(mae_red)

In [ ]:
revisar(lambda: red_precio.layers[-1].units == 1 and np.isclose(mae_red, mean_absolute_error(precio_real, np.exp(red_precio.predict(D_te, verbose=0).ravel()))) and mae_red < 600)

**Qué deberías ver:** un error medio cercano a **350 dólares**.

## Paso 3.2. Comparar

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

comparacion_precio = {
    "Regresión lineal": mean_absolute_error(precio_real, np.exp(LinearRegression().fit(D_tr, yd_tr).predict(D_te))),
    "Random forest": mean_absolute_error(precio_real, np.exp(RandomForestRegressor(200, random_state=42, n_jobs=-1).fit(D_tr, yd_tr).predict(D_te))),
    "Red neuronal": mae_red,
}
pd.Series(comparacion_precio, name="MAE (USD)").round(0)

**Qué deberías ver:** la regresión lineal con un error cercano a 1.600 dólares, y la red y el random forest cerca de 300 a 350. La relación entre las características y el precio **no es lineal**, y tanto la red (con sus capas ocultas) como el random forest la capturan. De nuevo, la red empata con un buen modelo clásico.

### Ejercicio 9
Usa `red_precio` para estimar el precio, **en dólares**, de un diamante de 1,0 quilates, corte `"Ideal"`, color `"G"`, pureza `"VS1"`, profundidad 61,5 y tabla 56. Guárdalo en `precio_estimado`. Pista: arma un DataFrame de una fila con las columnas de `F_D` y pásalo por `prep_d` y `esc_d`, en ese orden.

In [ ]:
precio_estimado = None  # TU CÓDIGO AQUÍ
print(precio_estimado)

In [ ]:
_d = pd.DataFrame([{"carat": 1.0, "cut": "Ideal", "color": "G", "clarity": "VS1", "depth": 61.5, "table": 56}])[F_D]
revisar(lambda: np.isclose(precio_estimado, float(np.exp(red_precio.predict(esc_d.transform(prep_d.transform(_d)), verbose=0)[0, 0])), rtol=1e-4))

---
# Parte 4. Caso: gama de precio (multiclase)

El mismo problema de la Clase 4: económica, media o premium, cada una con un tercio de los diamantes. Allá, el random forest llegó a cerca de 95% de exactitud.

In [ ]:
gama = pd.qcut(dm["price"], q=3, labels=[0, 1, 2]).astype(int).values      # 0 económica, 1 media, 2 premium
NOMBRES_GAMA = ["Económica", "Media", "Premium"]

Xg_tr, Xg_te, g_tr, g_te = train_test_split(dm[F_D], gama, test_size=0.2, random_state=42, stratify=gama)
esc_g = StandardScaler().fit(prep_d.fit_transform(Xg_tr))
G_tr, G_te = esc_g.transform(prep_d.transform(Xg_tr)), esc_g.transform(prep_d.transform(Xg_te))
print("Clases:", np.bincount(g_tr))

### Ejercicio 10
Construye, compila y entrena `red_gama`, igual que `red_precio` pero para **3 clases** con etiquetas **enteras**: elige la última capa y la pérdida correctas (Paso 1.1), con la métrica `"accuracy"`. Luego guarda en `exac_red_gama` la exactitud en prueba. Pista: la clase predicha es `argmax` de las probabilidades.

In [ ]:
keras.utils.set_random_seed(42)
red_gama = None        # TU CÓDIGO AQUÍ
exac_red_gama = None   # TU CÓDIGO AQUÍ
print(exac_red_gama)

In [ ]:
revisar(lambda: red_gama.layers[-1].units == 3 and red_gama.loss == "sparse_categorical_crossentropy"
        and np.isclose(exac_red_gama, accuracy_score(g_te, red_gama.predict(G_te, verbose=0).argmax(axis=1))) and exac_red_gama > 0.9)

**Qué deberías ver:** cerca de 0,95, igual que el random forest de la Clase 4.

### Ejercicio 11
Con la matriz de confusión de `red_gama`, ¿cuántos diamantes **económicos** confundió con **premium**? Guárdalo en `economica_como_premium`. Pista: `confusion_matrix(real, predicho)[fila, columna]`, donde la fila es la clase real.

In [ ]:
economica_como_premium = None  # TU CÓDIGO AQUÍ
print(economica_como_premium)

In [ ]:
revisar(lambda: economica_como_premium == confusion_matrix(g_te, red_gama.predict(G_te, verbose=0).argmax(axis=1))[0, 2])

**Qué deberías ver:** **0** o casi 0. Como en la Clase 4, los errores ocurren entre gamas vecinas, nunca entre los extremos.

---
# Parte 5. Caso: prendas de ropa con una red convolucional (láminas 191 a 212)

Aquí es donde las redes neuronales **sí** sacan ventaja. **Fashion-MNIST** tiene 70.000 imágenes de 28 × 28 pixeles de 10 tipos de prendas. Es más difícil que los dígitos de MNIST de la lámina 192.

## Paso 5.1. Cómo funciona una CNN

| Capa | Qué hace |
|---|---|
| `Conv2D(filtros, (3, 3))` | Desliza filtros de 3 × 3 por la imagen; cada filtro aprende a detectar un rasgo (bordes, texturas) |
| `MaxPooling2D((2, 2))` | Reduce la imagen a la mitad quedándose con el máximo de cada bloque de 2 × 2 |
| `Flatten()` | Convierte la imagen resultante en un vector |
| `Dense` | Combina los rasgos para decidir la clase |

**Parámetros de una `Conv2D`:** filtros × (alto × ancho × canales de entrada + 1). Por ejemplo, 32 filtros de 3 × 3 sobre una imagen de 1 canal: 32 × (9 + 1) = 320.

In [ ]:
(Xf_tr, yf_tr), (Xf_te, yf_te) = keras.datasets.fashion_mnist.load_data()
PRENDAS = ["Camiseta", "Pantalón", "Suéter", "Vestido", "Abrigo", "Sandalia", "Camisa", "Zapatilla", "Bolso", "Botín"]

fig, ejes = plt.subplots(1, 10, figsize=(13, 1.8))
for ax, k in zip(ejes, range(10)):
    i = np.where(yf_tr == k)[0][0]
    ax.imshow(Xf_tr[i], cmap="gray_r"); ax.set_title(PRENDAS[k], fontsize=8); ax.axis("off")
plt.show()

Xf_tr = (Xf_tr / 255.0).reshape(-1, 28, 28, 1).astype("float32")
Xf_te = (Xf_te / 255.0).reshape(-1, 28, 28, 1).astype("float32")
print("Entrenamiento:", Xf_tr.shape, "| Prueba:", Xf_te.shape)

**Qué deberías ver:** un ejemplo de cada prenda, y las formas (60000, 28, 28, 1) y (10000, 28, 28, 1): escaladas de 0 a 1 y con el canal agregado.

## Paso 5.2. Primero, una red densa (sin convolución)

Para comparar: una red que aplana la imagen y la trata como 784 columnas sueltas.

In [ ]:
keras.utils.set_random_seed(42)
densa_img = keras.Sequential([layers.Input(shape=(28, 28, 1)), layers.Flatten(),
                              layers.Dense(128, activation="relu"), layers.Dense(10, activation="softmax")])
densa_img.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
densa_img.fit(Xf_tr, yf_tr, epochs=5, batch_size=128, validation_split=0.1, verbose=0)
exac_densa = densa_img.evaluate(Xf_te, yf_te, verbose=0)[1]
print(f"Red densa: exactitud en prueba {exac_densa:.3f}")

**Qué deberías ver:** cerca de 0,87.

### Ejercicio 12
Construye `cnn_ropa` con esta arquitectura, compílala como la red densa y entrénala con los mismos parámetros (5 épocas, lotes de 128, 10% de validación). Guarda la exactitud en prueba en `exac_cnn`.

`Input(28, 28, 1)` → `Conv2D(32, (3, 3), relu)` → `MaxPooling2D((2, 2))` → `Conv2D(64, (3, 3), relu)` → `MaxPooling2D((2, 2))` → `Flatten()` → `Dropout(0.3)` → `Dense(10, softmax)`

In [ ]:
keras.utils.set_random_seed(42)
cnn_ropa = None   # TU CÓDIGO AQUÍ
exac_cnn = None   # TU CÓDIGO AQUÍ
print(exac_cnn)

In [ ]:
revisar(lambda: cnn_ropa.count_params() == 34826 and np.isclose(exac_cnn, cnn_ropa.evaluate(Xf_te, yf_te, verbose=0)[1]))

**Qué deberías ver:** cerca de **0,90 o 0,91**, mejor que la red densa con **menos parámetros** (34.826 contra más de 100.000). La convolución aprovecha que los pixeles cercanos están relacionados.

### Ejercicio 13
¿Qué par de prendas confunde más `cnn_ropa`? Calcula la matriz de confusión, ignora la diagonal (los aciertos) y encuentra la celda más grande. Guarda en `par_confundido` una tupla `(prenda_real, prenda_predicha)` con los **nombres** de `PRENDAS`.

In [ ]:
par_confundido = None  # TU CÓDIGO AQUÍ
print(par_confundido)

In [ ]:
_cm = confusion_matrix(yf_te, cnn_ropa.predict(Xf_te, verbose=0).argmax(axis=1)); np.fill_diagonal(_cm, 0)
_i, _j = np.unravel_index(_cm.argmax(), _cm.shape)
revisar(lambda: tuple(par_confundido) == (PRENDAS[_i], PRENDAS[_j]))

**Qué deberías ver:** casi siempre **camisa con camiseta** (o con suéter o abrigo). Tiene sentido: a 28 × 28 pixeles, son prendas muy parecidas incluso para una persona.

---
# Parte 6. Caso: pasajeros de avión con redes recurrentes (láminas 213 a 230)

## Paso 6.1. ¿Qué es una red recurrente?

Las redes de las Partes 2 a 5 reciben **todo de una vez**. Una **red recurrente (RNN)** recibe una **secuencia** y la lee **paso a paso**, guardando una memoria (el estado oculto $h$) que se actualiza en cada paso:

$$h_t = \\tanh(W \\cdot x_t + U \\cdot h_{t-1} + b)$$

Es decir: la memoria nueva combina el dato actual $x_t$ con la memoria anterior $h_{t-1}$. Al terminar la secuencia, $h$ resume todo lo visto.

**El problema:** en secuencias largas, la RNN simple olvida lo del principio (el gradiente que desaparece de la Clase 14).

**Las soluciones:**

- **LSTM:** agrega una "cinta" de memoria con tres puertas (olvido, entrada y salida) que deciden qué borrar, qué guardar y qué mostrar (lámina 214).
- **GRU:** una versión simplificada de la LSTM, con dos puertas. Tiene menos parámetros y suele rendir parecido.

| Capa de Keras | Puertas | Parámetros (con 32 neuronas y 1 entrada) |
|---|---|---|
| `SimpleRNN(32)` | Ninguna | 32 × (32 + 1 + 1) = 1.088 |
| `GRU(32)` | 2 | Cerca de 3 veces la RNN |
| `LSTM(32)` | 3 | 4 veces la RNN: 4.352 |

## Paso 6.2. Los datos y las ventanas

144 meses de pasajeros (miles), con tendencia y estacionalidad. Entrenamos con los primeros 120 meses y probamos con los últimos 24. Cada ejemplo: 12 meses → el mes siguiente.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

serie = sns.load_dataset("flights")["passengers"].values.astype("float32")
ENTRENO, VENTANA = 120, 12

esc_s = MinMaxScaler().fit(serie[:ENTRENO].reshape(-1, 1))
serie_e = esc_s.transform(serie.reshape(-1, 1)).ravel()

def ventanas(datos, largo):
    X, y = [], []
    for i in range(len(datos) - largo):
        X.append(datos[i:i + largo]); y.append(datos[i + largo])
    return np.array(X).reshape(-1, largo, 1), np.array(y)

S_tr, s_tr = ventanas(serie_e[:ENTRENO], VENTANA)
S_te, s_te = ventanas(serie_e[ENTRENO - VENTANA:], VENTANA)
print("Entrenamiento:", S_tr.shape, "(ejemplos, pasos de tiempo, características) | Prueba:", S_te.shape)

plt.figure(figsize=(10, 3)); plt.plot(serie); plt.axvline(ENTRENO, ls="--", color="gray")
plt.title("Pasajeros mensuales (miles): entrenamiento | prueba"); plt.show()

### Ejercicio 14
Comprueba la tabla del Paso 6.1: construye un modelo con `Input(shape=(12, 1))` y `SimpleRNN(32)`, y guarda en `params_rnn` los parámetros de la capa `SimpleRNN` (con `modelo.layers[0].count_params()`).

In [ ]:
params_rnn = None  # TU CÓDIGO AQUÍ
print(params_rnn)

In [ ]:
revisar(lambda: params_rnn == 32 * (32 + 1 + 1))

## Paso 6.3. Comparar RNN, LSTM y GRU

### Ejercicio 15
Completa `red_serie(capa)` para que arme, compile (`adam`, `mse`) y entrene (200 épocas, lotes de 16, `verbose=0`) un modelo con `Input(shape=(VENTANA, 1))`, la capa recibida y `Dense(1)`. Luego se entrenan las tres variantes y se comparan con la **línea base ingenua estacional** (repetir el mismo mes del año anterior).

In [ ]:
def red_serie(capa):
    keras.utils.set_random_seed(42)
    modelo = None   # TU CÓDIGO AQUÍ
    return modelo

real_s = serie[ENTRENO:]
predicciones_s = {"Ingenua estacional": serie[ENTRENO - 12:-12]}
for nombre, capa in [("RNN", layers.SimpleRNN(32)), ("LSTM", layers.LSTM(32)), ("GRU", layers.GRU(32))]:
    modelo = red_serie(capa)
    if modelo is not None:
        predicciones_s[nombre] = esc_s.inverse_transform(modelo.predict(S_te, verbose=0)).ravel()

rmse_s = {n: float(np.sqrt(mean_squared_error(real_s, p))) for n, p in predicciones_s.items()}
print({n: round(v, 1) for n, v in rmse_s.items()})

plt.figure(figsize=(10, 4))
plt.plot(range(len(serie)), serie, color="lightgray", label="real")
for n, p in predicciones_s.items():
    plt.plot(range(ENTRENO, len(serie)), p, label=f"{n} (RMSE {rmse_s[n]:.1f})")
plt.axvline(ENTRENO, ls="--", color="gray"); plt.legend(); plt.title("Predicción de los últimos 24 meses"); plt.show()

In [ ]:
revisar(lambda: set(rmse_s) == {"Ingenua estacional", "RNN", "LSTM", "GRU"})

**Qué deberías ver:** cuatro RMSE en miles de pasajeros. En nuestras pruebas, la línea base estuvo cerca de 50, y las redes entre 40 y 70 según el tipo; los números pueden variar un poco entre computadores.

**Lean la comparación con cuidado:** con solo 108 ventanas, **no siempre gana la red más compleja**, y a veces la línea base es muy difícil de superar. Las redes además predicen mal valores más altos que todos los que vieron en el entrenamiento (la tendencia sigue subiendo en la prueba).

### Ejercicio 16
Guarda en `mejor_serie` el nombre del modelo con menor RMSE, y en `supera_base` un booleano que diga si **alguna** red le ganó a la línea base.

In [ ]:
mejor_serie = None   # TU CÓDIGO AQUÍ
supera_base = None   # TU CÓDIGO AQUÍ
print(mejor_serie, supera_base)

In [ ]:
revisar(lambda: mejor_serie == min(rmse_s, key=rmse_s.get) and supera_base == any(v < rmse_s["Ingenua estacional"] for n, v in rmse_s.items() if n != "Ingenua estacional"))

---
# Parte 7. Desafío para la casa: superar el AUC de la fuga de clientes

En la Parte 2, la red y la logística quedaron cerca de 0,84 de ROC AUC. **El desafío: superar 0,85 en el conjunto de prueba**, sin tocar `A_te` ni `y_te` para entrenar.

Ideas para probar, de a una y midiendo con validación:

1. **Variables nuevas:** cargo total dividido por antigüedad, número de servicios contratados, contrato mes a mes con fibra (el segmento de la Clase 8).
2. **Arquitectura:** más o menos neuronas, más capas, más dropout, `BatchNormalization`.
3. **Entrenamiento:** tasa de aprendizaje de Adam (`keras.optimizers.Adam(learning_rate=...)`), tamaño de lote, paciencia.
4. **Otros modelos:** `GradientBoostingClassifier` o XGBoost (Clase 3), y comparar.

Registren en una tabla cada intento: qué cambiaron, AUC de validación y AUC de prueba al final. **Si nada supera claramente a la logística, esa también es una conclusión válida** para su informe: es justo lo que pasa en muchos problemas reales con datos en tabla.

In [ ]:
# Espacio para el desafío

---
# Cierre

| Caso | Mejor resultado | Conclusión |
|---|---|---|
| Fuga de clientes | Red y logística empatadas (AUC cerca de 0,84) | En tablas, probar primero modelos clásicos |
| Precio de diamantes | Red y random forest (MAE cerca de 300 a 350 USD), lineal muy atrás | Las capas ocultas capturan relaciones no lineales |
| Gama de precio | Red y random forest (cerca de 0,95) | Empate |
| Prendas de ropa | CNN (cerca de 0,91) sobre red densa (0,87) | Con imágenes, la convolución sí marca la diferencia |
| Pasajeros | Depende: la línea base es difícil de superar | Con pocos datos, comparar siempre con algo simple |

Guarden el notebook en `ejercicios_estudiantes` como `clase15_su_nombre.ipynb`.

---
# Soluciones

### Ejercicio 1
46 entradas × 32 neuronas + 32 sesgos = 1.504.

In [ ]:
params_capa1 = 46 * 32 + 32
print(params_capa1)

### Ejercicio 2
1.504 + (32 × 16 + 16) + (16 × 1 + 1) = 1.504 + 528 + 17 = 2.049. El Dropout no suma parámetros.

In [ ]:
params_total = (46 * 32 + 32) + (32 * 16 + 16) + (16 * 1 + 1)
_m = keras.Sequential([layers.Input(shape=(46,)), layers.Dense(32), layers.Dropout(0.3), layers.Dense(16), layers.Dense(1)])
print(params_total, _m.count_params())

### Ejercicio 3

In [ ]:
pesos_clase = {0: float(len(y_tr) / (2 * (y_tr == 0).sum())), 1: float(len(y_tr) / (2 * (y_tr == 1).sum()))}
print(pesos_clase)

### Ejercicio 4

In [ ]:
def construir_red_churn():
    keras.utils.set_random_seed(42)
    modelo = keras.Sequential([
        layers.Input(shape=(A_tr.shape[1],)),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    modelo.compile(optimizer="adam", loss="binary_crossentropy",
                   metrics=[keras.metrics.AUC(name="auc"), keras.metrics.Recall(name="recall")])
    return modelo

red_churn = construir_red_churn()
parada = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=10, restore_best_weights=True)
red_churn.fit(A_tr, y_tr, validation_data=(A_va, y_va), epochs=100, batch_size=64,
              class_weight=pesos_clase, callbacks=[parada], verbose=0)
prob_churn = red_churn.predict(A_te, verbose=0).ravel()
print(red_churn.count_params(), round(roc_auc_score(y_te, prob_churn), 3))

### Ejercicio 5

In [ ]:
recall_red = recall_score(y_te, prob_churn >= 0.5)
n_contactar = int((prob_churn >= 0.5).sum())
print(round(recall_red, 3), n_contactar)

### Ejercicio 6

In [ ]:
umbrales = np.round(np.arange(0.30, 0.81, 0.05), 2)
f1s = [f1_score(y_te, prob_churn >= u) for u in umbrales]
umbral_f1 = float(umbrales[int(np.argmax(f1s))])
print(umbral_f1, round(max(f1s), 3))

### Ejercicio 7

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(np.vstack([A_tr, A_va]), np.concatenate([y_tr, y_va]))
auc_logistica = roc_auc_score(y_te, lr.predict_proba(A_te)[:, 1])
gana = "red" if roc_auc_score(y_te, prob_churn) > auc_logistica else "logistica"
print(round(auc_logistica, 3), gana)

### Ejercicio 8

In [ ]:
keras.utils.set_random_seed(42)
red_precio = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),                                  # regresión: sin activación
])
red_precio.compile(optimizer="adam", loss="mse", metrics=["mae"])
red_precio.fit(D_tr, yd_tr, validation_split=0.2, epochs=200, batch_size=64,
               callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)
mae_red = mean_absolute_error(precio_real, np.exp(red_precio.predict(D_te, verbose=0).ravel()))
print(round(mae_red, 1))

### Ejercicio 9
El DataFrame tiene que tener las columnas en el mismo orden que `F_D`, y pasar por los mismos dos transformadores que el entrenamiento.

In [ ]:
nuevo = pd.DataFrame([{"carat": 1.0, "cut": "Ideal", "color": "G", "clarity": "VS1", "depth": 61.5, "table": 56}])[F_D]
precio_estimado = float(np.exp(red_precio.predict(esc_d.transform(prep_d.transform(nuevo)), verbose=0)[0, 0]))
print(round(precio_estimado))

### Ejercicio 10
Para 3 clases con etiquetas enteras: `Dense(3, activation="softmax")` y `sparse_categorical_crossentropy`.

In [ ]:
keras.utils.set_random_seed(42)
red_gama = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax"),
])
red_gama.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
red_gama.fit(G_tr, g_tr, validation_split=0.2, epochs=200, batch_size=64,
             callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)], verbose=0)
exac_red_gama = accuracy_score(g_te, red_gama.predict(G_te, verbose=0).argmax(axis=1))
print(round(exac_red_gama, 3))

### Ejercicio 11
Fila 0 (real: económica), columna 2 (predicha: premium).

In [ ]:
economica_como_premium = confusion_matrix(g_te, red_gama.predict(G_te, verbose=0).argmax(axis=1))[0, 2]
print(economica_como_premium)

### Ejercicio 12
Parámetros: 320 (primera Conv2D) + 18.496 (segunda) + 16.010 (Dense: 5 × 5 × 64 = 1.600 entradas × 10 + 10) = 34.826.

In [ ]:
keras.utils.set_random_seed(42)
cnn_ropa = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(10, activation="softmax"),
])
cnn_ropa.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn_ropa.fit(Xf_tr, yf_tr, epochs=5, batch_size=128, validation_split=0.1, verbose=0)
exac_cnn = cnn_ropa.evaluate(Xf_te, yf_te, verbose=0)[1]
print(cnn_ropa.count_params(), round(exac_cnn, 3))

### Ejercicio 13
`np.fill_diagonal(cm, 0)` borra los aciertos; `np.unravel_index(cm.argmax(), cm.shape)` da la fila y la columna de la celda más grande.

In [ ]:
cm = confusion_matrix(yf_te, cnn_ropa.predict(Xf_te, verbose=0).argmax(axis=1))
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
par_confundido = (PRENDAS[i], PRENDAS[j])
print(par_confundido, cm[i, j])

### Ejercicio 14
32 neuronas × (32 de la memoria anterior + 1 entrada + 1 sesgo) = 1.088.

In [ ]:
_m = keras.Sequential([layers.Input(shape=(12, 1)), layers.SimpleRNN(32), layers.Dense(1)])
params_rnn = _m.layers[0].count_params()
print(params_rnn)

### Ejercicio 15

In [ ]:
def red_serie(capa):
    keras.utils.set_random_seed(42)
    modelo = keras.Sequential([layers.Input(shape=(VENTANA, 1)), capa, layers.Dense(1)])
    modelo.compile(optimizer="adam", loss="mse")
    modelo.fit(S_tr, s_tr, epochs=200, batch_size=16, verbose=0)
    return modelo

real_s = serie[ENTRENO:]
predicciones_s = {"Ingenua estacional": serie[ENTRENO - 12:-12]}
for nombre, capa in [("RNN", layers.SimpleRNN(32)), ("LSTM", layers.LSTM(32)), ("GRU", layers.GRU(32))]:
    predicciones_s[nombre] = esc_s.inverse_transform(red_serie(capa).predict(S_te, verbose=0)).ravel()
rmse_s = {n: float(np.sqrt(mean_squared_error(real_s, p))) for n, p in predicciones_s.items()}
print({n: round(v, 1) for n, v in rmse_s.items()})

### Ejercicio 16

In [ ]:
mejor_serie = min(rmse_s, key=rmse_s.get)
supera_base = any(v < rmse_s["Ingenua estacional"] for n, v in rmse_s.items() if n != "Ingenua estacional")
print(mejor_serie, supera_base)